# 02 — Evaluation Data Preparation

Freeze reproducible calibration exploration/confirmation and development scenarios for Round 2. Preserve original roles, IDs, boxes, negative images and geometry; label synthetic conditions explicitly. No policy search, training or inference runs.

- **Inputs:** Notebook 01 saved audit, frozen protocol and original BDD sources.
- **Outputs:** `docs/round2-evaluation-v1/`: immutable scenario manifest, compressed membership, support, sample diagnostics and visual evidence. Images are prepared lazily in memory.
- **Prerequisites:** Project study kernel and saved notebook 01 artifacts. No final-test access.

**Outline**

1. [Configuration and Preflight](#configuration-preflight)
2. [Frozen Sets and Support](#frozen-sets)
3. [Clean and Synthetic Scenarios](#synthetic-scenarios)
4. [Preparation and Handoff Summary](#preparation-summary)

Set paths, then **Restart & Run All**. Notebook 01 is consumed through its saved audit only. Missing/stale prerequisites fail without launching upstream phases.

<a id="configuration-preflight"></a>

## 1. Configuration and Preflight

In [ ]:
import gzip
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vehicle_detection.common import digest, file_hash, freeze_json, read_json, require_paths
from vehicle_detection.evaluation_data import evaluation_sets, scenario_image, validate_audit
from vehicle_detection.five_class import FIVE_CLASSES, support_audit
from vehicle_detection.protocol import validate_protocol
from vehicle_detection.stress import diagnostics

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA_ROOT = ROOT / "data/bdd100k"
PROTOCOL_PATH = ROOT / "runs/issue5-v1/protocol.json"
AUDIT_PATH = ROOT / "docs/round2-data-v1/audit.json"
OUTPUT = ROOT / "docs/round2-evaluation-v1"
SEED = 2026
QUALITY_SAMPLE_PER_SET = 4
require_paths(data=DATA_ROOT, protocol=PROTOCOL_PATH, data_audit=AUDIT_PATH)
for protected in (DATA_ROOT, PROTOCOL_PATH.parent, AUDIT_PATH.parent):
    a, b = OUTPUT.resolve(), protected.resolve()
    if a == b or a in b.parents or b in a.parents:
        raise ValueError(f"Output overlaps preserved data: {protected}")
if OUTPUT.exists() and any(OUTPUT.iterdir()) and not (OUTPUT / "evaluation_sets.json").is_file():
    raise ValueError("Unknown evaluation directory; choose a new output root")
if type(QUALITY_SAMPLE_PER_SET) is not int or QUALITY_SAMPLE_PER_SET < 1:
    raise ValueError("Quality sample size must be a positive integer")
cv2.setNumThreads(1)
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})
protocol = validate_protocol(read_json(PROTOCOL_PATH))
audit = validate_audit(read_json(AUDIT_PATH), protocol)
manifest = evaluation_sets(protocol, seed=SEED)
manifest["data_audit_id"] = audit["audit_id"]
manifest["quality_sample_per_set"] = QUALITY_SAMPLE_PER_SET
manifest["manifest_id"] = digest({k: v for k, v in manifest.items() if k != "manifest_id"})
OUTPUT.mkdir(parents=True, exist_ok=True)
freeze_json(OUTPUT / "evaluation_sets.json", manifest)

<a id="frozen-sets"></a>

## 2. Frozen Sets and Support

All calibration groups are ordered by SHA256(seed, group) and split into two equal group halves. This partitions only the existing calibration role: all four top-level roles remain unchanged. Confirmation is reserved for shortlisted Round 2 choices; historical v2 already used calibration data, so this is not a previously untouched holdout. Development retains every frozen ID and is evaluation-only. Inspect rare/zero support without rebalancing.

In [ ]:
lookup = {
    row["id"]: row
    for role, rows in protocol["splits"].items()
    if role != "final_test"
    for row in rows
}
sets = {name: [lookup[i] for i in ids] for name, ids in manifest["sets"].items()}
group_sets = {name: {row["group"] for row in rows} for name, rows in sets.items()}
for name, groups in group_sets.items():
    assert all(not groups.intersection(other) for key, other in group_sets.items() if key != name)
assert set(manifest["sets"]["calibration_exploration"]) | set(
    manifest["sets"]["calibration_confirmation"]
) == {row["id"] for row in protocol["splits"]["calibration"]}
assert set(manifest["sets"]["development_validation"]) == {
    row["id"] for row in protocol["splits"]["development_validation"]
}
support = (
    pd.DataFrame({name: support_audit(rows) for name, rows in sets.items()}).fillna(0).astype(int)
)
support.to_csv(OUTPUT / "set_support.csv")
keys = ["images", "new_negative_images", *[f"new_presence:{c}" for c in FIVE_CLASSES]]
display(support.reindex(keys, fill_value=0).T)
print("Groups:", {name: len(groups) for name, groups in group_sets.items()})
print("Class support <5 cannot support separate subgroup inference; no oversampling.")
membership = pd.DataFrame(
    [
        {
            "evaluation_set": name,
            "role": "calibration" if name.startswith("calibration") else name,
            **{
                k: row[k]
                for k in (
                    "id",
                    "group",
                    "source_path",
                    "sha256",
                    "labels_sha256",
                    "width",
                    "height",
                )
            },
        }
        for name, rows in sets.items()
        for row in rows
    ]
).sort_values(["evaluation_set", "id"])
content = gzip.compress(membership.to_csv(index=False).encode(), mtime=0)
path = OUTPUT / "membership.csv.gz"
if path.exists() and path.read_bytes() != content:
    raise ValueError("Evaluation membership changed; choose a new output root")
if not path.exists():
    path.write_bytes(content)

<a id="synthetic-scenarios"></a>

## 3. Clean and Synthetic Scenarios

The same IDs and five-class labels apply to every condition: clean plus synthetic underexposure (low light), shot/read noise and motion blur at fixed severities 1–3. Parameters reuse `stress.corrupt`; the per-image random seed depends on the recorded seed, ID, family and severity. Always start from original sources and encode JPEG95 once, including clean. No composed corruption, restoration policy or bulk dataset is created. Natural weather/time/scene metadata stays descriptive and is not renamed as synthetic.

In [ ]:
display(pd.DataFrame(manifest["scenarios"]))
print(manifest["pixel_pipeline"])
quality_rows, examples = [], {}
for name, rows in sets.items():
    sample = sorted(rows, key=lambda r: (digest([SEED, r["id"]]), r["id"]))[:QUALITY_SAMPLE_PER_SET]
    for row in sample:
        for scenario in manifest["scenarios"]:
            image = scenario_image(row, DATA_ROOT, scenario, seed=SEED)
            quality_rows.append(
                {
                    "evaluation_set": name,
                    "id": row["id"],
                    "scenario": scenario["id"],
                    "synthetic": scenario["synthetic"],
                    **diagnostics(image),
                }
            )
            if row == sample[0] and scenario["severity"] in (0, 2):
                examples[name, scenario["id"]] = image
quality = pd.DataFrame(quality_rows)
quality.to_csv(OUTPUT / "scenario_quality_sample.csv", index=False)
display(quality.groupby("scenario")[["brightness", "sharpness", "noise_highpass"]].median())
shown = [s for s in manifest["scenarios"] if s["severity"] in (0, 2)]
fig, axes = plt.subplots(len(sets), len(shown), figsize=(16, 9), squeeze=False)
for row_axes, name in zip(axes, sets, strict=True):
    for ax, scenario in zip(row_axes, shown, strict=True):
        ax.imshow(cv2.cvtColor(examples[name, scenario["id"]], cv2.COLOR_BGR2RGB))
        ax.set_title(f"{name}\n{scenario['id']}", fontsize=8)
        ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT / "scenario_examples.png")
plt.show()
print(
    "Synthetic visual checks are not evidence of detection improvement "
    "or real low-light robustness."
)

<a id="preparation-summary"></a>

## 4. Preparation and Handoff Summary

In [ ]:
summary = {
    "manifest_id": manifest["manifest_id"],
    "data_audit_id": audit["audit_id"],
    "set_sizes": {name: len(rows) for name, rows in sets.items()},
    "scenario_count": len(manifest["scenarios"]),
    "membership_sha256": file_hash(OUTPUT / "membership.csv.gz"),
    "sample_diagnostics_sha256": file_hash(OUTPUT / "scenario_quality_sample.csv"),
    "quality_sample_per_set": QUALITY_SAMPLE_PER_SET,
    "artifacts": {"protocol": str(PROTOCOL_PATH), "data_audit": str(AUDIT_PATH)},
    "status": "data_prepared_no_policy_selection_or_model_evaluation",
}
freeze_json(OUTPUT / "summary.json", summary)
display(pd.Series(summary["set_sizes"], name="frozen images"))
print("Manifest for issue 30:", OUTPUT / "evaluation_sets.json")
print(
    "Use frozen protocol rows and scenario_image for checked, lazy preparation; "
    "retain existing five-class mapping."
)
print("\n".join(manifest["limitations"]))